In [14]:
import io
import os
import pandas as pd

# The raw output string extracted by your layout engine
html_table_string = """
<table>\n  <thead>\n    <tr>\n        <th> </th>\n        <th colspan=\"3\">Nine Months Ended September 30, 2024</th>\n    </tr>\n    <tr>\n        <th> </th>\n        <th> </th>\n        <th>Adjustments from Adoption of the New Crypto Assets Standard</th>\n        <th> </th>\n    </tr>\n    <tr>\n        <th>Consolidated Statement of Operations (unaudited):</th>\n        <th>As Previously Reported</th>\n        <th>Standard</th>\n        <th>As Adjusted</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n        <td>Other (expense) income, net</td>\n        <td>$ (142)</td>\n        <td>$ 242</td>\n        <td>$ 100</td>\n    </tr>\n    <tr>\n        <td>Provision for income taxes</td>\n        <td>$ 1,403</td>\n        <td>$ 53</td>\n        <td>$ 1,456</td>\n    </tr>\n    <tr>\n        <td>Net income attributable to common stockholders</td>\n        <td>$ 4,774</td>\n        <td>$ 189</td>\n        <td>$ 4,963</td>\n    </tr>\n    <tr>\n        <td colspan=\"4\">Net income per share attributable to common stockholders:</td>\n    </tr>\n    <tr>\n        <td>Basic</td>\n        <td>$ 1.51</td>\n        <td>$ 0.06</td>\n        <td>$ 1.57</td>\n    </tr>\n    <tr>\n        <td>Diluted</td>\n        <td>$ 1.38</td>\n        <td>$ 0.05</td>\n        <td>$ 1.43</td>\n    </tr>\n  </tbody>\n</table>
"""

def save_html_table_to_csv(html_string, output_directory, table_id):
    os.makedirs(output_directory, exist_ok=True)
    
    try:
        # 1. HTML to Pandas DataFrame
        df_list = pd.read_html(io.StringIO(html_string), flavor="lxml")
        df = df_list[0]
        
        # 2. Flatten and filter out "Unnamed:" artifacts from headers
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = ["_".join(str(level).strip() for level in col if "Unnamed:" not in str(level)) for col in df.columns]
        else:
            df.columns = [str(col).strip() if "Unnamed:" not in str(col) else "" for col in df.columns]
            
        # Give a clean name if the first column header became completely empty
        df.columns = [f"Line_Item_{i}" if col == "" else col for i, col in enumerate(df.columns)]
        
        # Clean up general spacing inside the cells
        df = df.map(lambda x: " ".join(str(x).split()) if pd.notna(x) else x)
        
        # 3. Save to pristine CSV file
        csv_filename = f"{table_id}.csv"
        csv_path = os.path.join(output_directory, csv_filename)
        df.to_csv(csv_path, index=False)
        
        print(f"Successfully saved: {csv_path}")
        return csv_path, df
        
    except Exception as e:
        print(f"Error processing table {table_id}: {str(e)}")
        return None, None

# --- RUNNING THE CODE ---
output_dir = "./data/extracted_csvs"
t_id = "table_sec_01"

csv_path, dataframe_object = save_html_table_to_csv(html_table_string, output_dir, t_id)
# print(dataframe_object.to_markdown(index=False))

Successfully saved: ./data/extracted_csvs/table_sec_01.csv


In [15]:
# Just type the variable name alone at the end of a cell
dataframe_object

,Consolidated Statement of Operations (unaudited):,"Nine Months Ended September 30, 2024_As Previously Reported","Nine Months Ended September 30, 2024_Adjustments from Adoption of the New Crypto Assets Standard_Standard","Nine Months Ended September 30, 2024_As Adjusted"
0,"Other (expense) income, net",$ (142),$ 242,$ 100
1,Provision for income taxes,"$ 1,403",$ 53,"$ 1,456"
2,Net income attributable to common stockholders,"$ 4,774",$ 189,"$ 4,963"
3,Net income per share attributable to common st...,Net income per share attributable to common st...,Net income per share attributable to common st...,Net income per share attributable to common st...
4,Basic,$ 1.51,$ 0.06,$ 1.57
5,Diluted,$ 1.38,$ 0.05,$ 1.43
